In [1]:
!pip install scanpy pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 24.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import scanpy as sc

#### NORMAN PERTURBENCH DATA ####

# Load the .h5ad file
adata = sc.read_h5ad('norman19_processed.h5ad')

print("Observation (cell) metadata columns in adata.obs:")
print(adata.obs.columns.tolist())

# Print all variable (gene) metadata columns
print("\nVariable (gene) metadata columns in adata.var:")
print(adata.var.columns.tolist())

Observation (cell) metadata columns in adata.obs:
['guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo', 'condition', 'cov_merged', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes']

Variable (gene) metadata columns in adata.var:
['ensemble_id', 'ncounts', 'ncells', 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches']


In [3]:
import pandas as pd

# View the unique values of 'perturbation_type' and 'perturbation' in the metadata
print("Unique values in 'perturbation_type':")
print(adata.obs['perturbation_type'].unique())

pd.set_option('display.max_rows', None)

print("\nUnique values in 'perturbation':")
print(adata.obs['perturbation'].cat.categories.tolist())  # Use .cat.categories for categorical data

# If you want to see a few sample values for both columns
print("\nSample rows for 'perturbation_type' and 'perturbation':")
print(adata.obs[['perturbation_type', 'perturbation']].head())

Unique values in 'perturbation_type':
['CRISPR']
Categories (1, object): ['CRISPR']

Unique values in 'perturbation':
['AHR', 'AHR+FEV', 'AHR+KLF1', 'ARID1A', 'ARRDC3', 'ATL1', 'BAK1', 'BCL2L11', 'BCL2L11+BAK1', 'BCL2L11+TGFBR2', 'BCORL1', 'BPGM', 'BPGM+SAMD1', 'BPGM+ZBTB1', 'C19orf26', 'C3orf72', 'C3orf72+FOXL2', 'CBFA2T3', 'CBL', 'CBL+CNN1', 'CBL+PTPN12', 'CBL+PTPN9', 'CBL+TGFBR2', 'CBL+UBASH3A', 'CBL+UBASH3B', 'CDKN1A', 'CDKN1B', 'CDKN1B+CDKN1A', 'CDKN1C', 'CDKN1C+CDKN1A', 'CDKN1C+CDKN1B', 'CEBPA', 'CEBPB', 'CEBPB+CEBPA', 'CEBPB+MAPK1', 'CEBPB+OSR2', 'CEBPB+PTPN12', 'CEBPE', 'CEBPE+CEBPA', 'CEBPE+CEBPB', 'CEBPE+CNN1', 'CEBPE+KLF1', 'CEBPE+PTPN12', 'CEBPE+RUNX1T1', 'CEBPE+SPI1', 'CELF2', 'CITED1', 'CKS1B', 'CLDN6', 'CNN1', 'CNN1+MAPK1', 'CNN1+UBASH3A', 'CNNM4', 'COL1A1', 'COL2A1', 'CSRNP1', 'DLX2', 'DUSP9', 'DUSP9+ETS2', 'DUSP9+IGDCC3', 'DUSP9+KLF1', 'DUSP9+MAPK1', 'DUSP9+PRTG', 'DUSP9+SNAI1', 'EGR1', 'ELMSAN1', 'ETS2', 'ETS2+CEBPE', 'ETS2+CNN1', 'ETS2+IGDCC3', 'ETS2+IKZF3', 'ETS2+MA

In [4]:
#### TESTING TO SEE IF THE PERTURBATION METADATA IS CORRECTLY INPUTTED TO DATAMODULE ####

!pip install datasets

from datasets import load_from_disk

tgt_dataset = load_from_disk('/lustre/scratch126/cellgen/team205/bair/perturbench/perturbench_data/norman/dataset_all_tgt_random_pairing/perturbation.dataset')
tgt_dataset = tgt_dataset.select(indices=[100])
perturbation_info = tgt_dataset[0].get('perturbation', None)

if perturbation_info:
    if isinstance(perturbation_info, str):
        perturbation_info = perturbation_info.split('+')  # Split by '+'
    elif isinstance(perturbation_info, list):
        # Already a list, ensure splitting for each element
        perturbation_info = [gene for genes in perturbation_info for gene in genes.split('+')]

print(perturbation_info)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 24.2
[notice] To update, run: python3 -m pip install --upgrade pip
['ETS2', 'MAP7D1']


In [10]:
import pickle

# Total number of cells
total_cells = adata.shape[0]
print(f"Total number of cells: {total_cells}")

with open('/lustre/scratch126/cellgen/team205/bair/perturbench/perturbench_data/norman/token_id_to_genename_all.pkl', 'rb') as file:
    token_to_gene_dict = pickle.load(file)

    
def get_token_ids_for_genes(genes, token_to_gene_dict):
    """
    Retrieve token IDs for a list of gene names or a single concatenated gene string.

    Args:
        genes (list of str or str): List of gene names or a single concatenated string of gene names.
        token_to_gene_dict (dict): Dictionary mapping token IDs to gene names.

    Returns:
        list of int: List of token IDs corresponding to the input gene names.
    """
    # Convert the dictionary to map gene names to token IDs for easy lookup
    gene_to_token = {v: k for k, v in token_to_gene_dict.items()}

    # If input is a string, split by '+' to handle combinations of genes
    if isinstance(genes, str):
        genes = genes.split('+')

    # Look up each gene in the dictionary and retrieve the token IDs
    token_ids = [gene_to_token[gene] for gene in genes if gene in gene_to_token]

    return token_ids

# Number of cells with the perturbation value 'control'
control_cells = adata.obs[adata.obs['perturbation'] == 'control'].shape[0]
print(f"Number of cells with perturbation value 'control': {control_cells}")

# for perturbation in adata.obs['perturbation'].cat.categories.tolist():
#     # Number of cells with the perturbation value 'control'
#     cells_of_interest = adata.obs[adata.obs['perturbation'] == perturbation].shape[0]
#     print(f"Number of cells with perturbation value '{perturbation}': {cells_of_interest}")

for perturbation in adata.obs['perturbation'].cat.categories.tolist():
    # Number of cells with the perturbation value 'control'
    corresponding_token = get_token_ids_for_genes(perturbation, token_to_gene_dict)
    print(f"Perturbation '{perturbation}' corresponds to token {corresponding_token}")

Total number of cells: 111445
Number of cells with perturbation value 'control': 11855
Perturbation 'AHR' corresponds to token [51]
Perturbation 'AHR+FEV' corresponds to token [51]
Perturbation 'AHR+KLF1' corresponds to token [51, 3626]
Perturbation 'ARID1A' corresponds to token [2910]
Perturbation 'ARRDC3' corresponds to token [3159]
Perturbation 'ATL1' corresponds to token [748]
Perturbation 'BAK1' corresponds to token [1003]
Perturbation 'BCL2L11' corresponds to token [154]
Perturbation 'BCL2L11+BAK1' corresponds to token [154, 1003]
Perturbation 'BCL2L11+TGFBR2' corresponds to token [154, 386]
Perturbation 'BCORL1' corresponds to token [2002]
Perturbation 'BPGM' corresponds to token [2768]
Perturbation 'BPGM+SAMD1' corresponds to token [2768, 2219]
Perturbation 'BPGM+ZBTB1' corresponds to token [2768, 3758]
Perturbation 'C19orf26' corresponds to token []
Perturbation 'C3orf72' corresponds to token []
Perturbation 'C3orf72+FOXL2' corresponds to token [1510]
Perturbation 'CBFA2T3' co

In [6]:
# print(adata.obs[['good_coverage', 'cell_type', 'perturbation_type', 'celltype', 'perturbation', 'nperts', 'ngenes', 'condition']].sample(n=20))

true_coverage = adata.obs[adata.obs['good_coverage'] == True].shape[0]
print(f"Number of cells with good coverage: {true_coverage}")

false_coverage = adata.obs[adata.obs['good_coverage'] == False].shape[0]
print(f"Number of cells with bad coverage: {false_coverage}")

# Number of cells with 'perturbation' = 'control' and 'good_coverage' = True
good_coverage_control = adata.obs[(adata.obs['perturbation'] == 'control') & (adata.obs['good_coverage'] == True)].shape[0]
print(f"Number of cells with 'control' perturbation and good coverage: {good_coverage_control}")

# Number of cells with 'perturbation' = 'control' and 'good_coverage' = False
bad_coverage_control = adata.obs[(adata.obs['perturbation'] == 'control') & (adata.obs['good_coverage'] == False)].shape[0]
print(f"Number of cells with 'control' perturbation and bad coverage: {bad_coverage_control}")

Number of cells with good coverage: 104507
Number of cells with bad coverage: 6938
Number of cells with 'control' perturbation and good coverage: 11183
Number of cells with 'control' perturbation and bad coverage: 672


In [7]:
#### CYTOIMGEN_PROCESSED CELLGEN DATA ####

# Load the .h5ad file
adata2 = sc.read_h5ad('/lustre/scratch126/cellgen/team205/av13/PETRA/data/h5d_files/cytoimmgen_processed.h5ad')
        
print("Observation (cell) metadata columns in adata.obs:")
print(adata2.obs.columns.tolist())

# Print all variable (gene) metadata columns
print("\nVariable (gene) metadata columns in adata.var:")
print(adata2.var.columns.tolist())

Observation (cell) metadata columns in adata.obs:
['Age', 'Cell_culture_batch', 'Cell_state', 'Cell_type', 'Donor', 'Sex', 'Time_point', 'batch', 'S_score', 'G2M_score', 'Phase', 'Activation_level', 'Cell_population']

Variable (gene) metadata columns in adata.var:
['gene_name', 'highly_variable', 'means', 'dispersions', 'dispersions_norm']


In [8]:
#### EB PAIRED CELLGEN DATA ####

# Load the .h5ad file
adata3 = sc.read_h5ad('/lustre/scratch126/cellgen/team205/bair/cg/T_perturb/T_perturb/pp/res/eb/h5ad_pairing_all_tgt_random_pairing/1_Day 06-09.h5ad')
        
print("Observation (cell) metadata columns in adata.obs:")
print(adata3.obs.columns.tolist())

# Print all variable (gene) metadata columns
print("\nVariable (gene) metadata columns in adata.var:")
print(adata3.var.columns.tolist())

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/lustre/scratch126/cellgen/team205/bair/cg/T_perturb/T_perturb/pp/res/eb/h5ad_pairing_all_tgt_random_pairing/1_Day 06-09.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:


import pickle

# Replace 'your_pickle_file.pkl' with the path to your pickle file
pickle_file_path = '/lustre/scratch126/cellgen/team205/bair/perturbench/perturbench_data/norman/token_id_to_genename_all.pkl'

# Open the pickle file in binary read mode
with open(pickle_file_path, 'rb') as file:
    data = pickle.load(file)  # Load the content of the pickle file
    
    print(data[42])
    
    gene_list = ['ABI3BP']  # Replace with your actual list of gene names

    # Create a reverse dictionary for quick lookup of indices by gene names
    idx_for_gene = {v: k for k, v in data.items()}

    # Get a list of indices (IDs) for the given list of gene names using list comprehension
    idx_list = [idx_for_gene.get(gene_name) for gene_name in gene_list]

    print(idx_list)

# Check the type of data to handle it appropriately
if isinstance(data, list):
    # If it's a list, print the first 10 items
    for line in data[:10]:
        print(line)
elif isinstance(data, dict):
    # If it's a dictionary, print the first 10 key-value pairs
    for i, (key, value) in enumerate(data.items()):
        print(f"{key}: {value}")
        if i >= 9:  # Only print the first 10 items
            break
else:
    # If it's another type, print its first 10 lines or elements
    print(str(data)[:1000])  # Adjust the slice if necessary

In [ ]:


import pickle

# Replace 'your_pickle_file.pkl' with the path to your pickle file
pickle_file_path = '/lustre/scratch126/cellgen/team205/bair/perturbench/perturbench_data/norman/tokenid_to_rowid_all.pkl'

# Open the pickle file in binary read mode
with open(pickle_file_path, 'rb') as file:
    data = pickle.load(file)  # Load the content of the pickle file

# Check the type of data to handle it appropriately
if isinstance(data, list):
    # If it's a list, print the first 10 items
    for line in data[:10]:
        print(line)
elif isinstance(data, dict):
    # If it's a dictionary, print the first 10 key-value pairs
    for i, (key, value) in enumerate(data.items()):
        print(f"{key}: {value}")
        if i >= 9:  # Only print the first 10 items
            break
else:
    # If it's another type, print its first 10 lines or elements
    print(str(data)[:1000])  # Adjust the slice if necessary

In [11]:
import scanpy as sc

# Load the problematic .h5ad file
adata_subset = sc.read_h5ad('/lustre/scratch126/cellgen/team205/bair/perturbench/perturbench_data/norman/h5ad_pairing_all/norman_all.h5ad')

# Check if 'ensembl_id' is in adata.var.columns
if 'ensembl_id' in adata_subset.var.columns:
    # Compare index and 'ensembl_id' column
    index_ensembl_ids = adata_subset.var_names.tolist()
    column_ensembl_ids = adata_subset.var['ensembl_id'].tolist()
    
    # Identify discrepancies
    discrepancies = [i for i, (idx, col) in enumerate(zip(index_ensembl_ids, column_ensembl_ids)) if idx != col]
    
    print(f"Number of discrepancies between index and 'ensembl_id' column: {len(discrepancies)}")
    
    if discrepancies:
        print("Sample discrepancies:")
        for i in discrepancies[:10]:  # Show first 10 discrepancies
            print(f"Index: {index_ensembl_ids[i]}, Column: {column_ensembl_ids[i]}")
    else:
        print("No discrepancies found between index and 'ensembl_id' column.")
else:
    print("'ensembl_id' column not found in adata.var.")

'ensembl_id' column not found in adata.var.
